# Canales: tres formas de describir un color

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Nomenclatura oficial:** Procesamiento de Imágenes
* **Nombre de trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

## Objetivo

Descomponer una imagen en sus canales y describir su color de tres maneras con OpenCV —RGB, escala de grises y HSV—, prestando atención a las escalas numéricas que la diferencian de scikit-image.

## Al terminar este cuaderno van a poder:

1. Convertir una imagen a escala de grises con `cv2.cvtColor` y explicar por qué no coincide exactamente con `color.rgb2gray`.
2. Separar y volver a unir los canales de una imagen con `cv2.split` y `cv2.merge`.
3. Explicar qué representa cada canal del espacio HSV.
4. Traducir un umbral escrito en la escala de scikit-image a la escala de OpenCV.
5. Armar una figura que compare las tres representaciones de una misma imagen.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Canal** | Cada una de las capas de números que forman una imagen. |
| **Escala de grises** | Una representación con un solo canal que conserva la luz y descarta el color. |
| **Espacio de color** | Una forma de describir numéricamente los colores: RGB, HSV y otros. |
| **Matiz (H)** | El "tipo" de color: rojo, amarillo, azul. Se mide como una posición en un círculo. |
| **Saturación (S)** | Qué tan puro es el color: de gris apagado a color intenso. |
| **Valor (V)** | Cuánta luz tiene el color: de negro a brillante. |

### ⚙ Configuración del entorno

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import skimage
from skimage import color

from utils import mostrar_imágenes

print(f"OpenCV {cv2.__version__} — NumPy {np.__version__} — scikit-image {skimage.__version__}")

Retomamos la función `leer_imagen_rgb` que escribimos en el cuaderno anterior. La volvemos a definir acá para que este cuaderno pueda ejecutarse de forma independiente.

In [ ]:
CARPETA_IMÁGENES = Path("..") / "imagenes"

def leer_imagen_rgb(ruta_de_la_imagen):
    """Lee una imagen con OpenCV, verifica la lectura y la devuelve en RGB."""
    imagen_leída = cv2.imread(filename=str(ruta_de_la_imagen), flags=cv2.IMREAD_COLOR)
    assert imagen_leída is not None, f"No se pudo leer: {ruta_de_la_imagen}"
    imagen_convertida = cv2.cvtColor(src=imagen_leída, code=cv2.COLOR_BGR2RGB)
    return imagen_convertida

imagen_globos = leer_imagen_rgb(CARPETA_IMÁGENES / "globos.jpg")
forma_globos = imagen_globos.shape
print(f"Imagen de los globos cargada: {forma_globos}")

La imagen de los globos tiene 1024 filas, 1024 columnas y 3 canales, ya ordenados en RGB. La elegimos porque combina un fondo de un solo color —el cielo— con objetos de colores muy distintos, lo que permite ver con claridad qué aporta cada canal.

## 1. Escala de grises

**Analogía.** Una foto en blanco y negro de un diario conserva dónde hay luz y dónde hay sombra, pero descarta el color. Alcanza para reconocer una cara o leer un titular, y ocupa mucho menos tinta.

**Bajada tecnológica.** Muchos algoritmos de visión trabajan en grises: la lectura de texto (OCR), la detección de bordes y los detectores clásicos de rostros. Con un solo canal procesan un tercio de los datos y no dependen del color de la iluminación.

### ✎ Antes de ejecutar

Al convertir a grises, ¿cuántas dimensiones va a tener el arreglo resultante? ¿Qué rango de valores esperan: de 0 a 1 o de 0 a 255?

In [ ]:
# 'COLOR_RGB2GRAY' combina los tres canales en uno solo de intensidad.
globos_gris_opencv = cv2.cvtColor(src=imagen_globos, code=cv2.COLOR_RGB2GRAY)

forma_gris = globos_gris_opencv.shape
tipo_gris = globos_gris_opencv.dtype
mínimo_gris = globos_gris_opencv.min()
máximo_gris = globos_gris_opencv.max()

print(f"Forma: {forma_gris} | Tipo: {tipo_gris} | Rango: {mínimo_gris} a {máximo_gris}")

El resultado tiene dos dimensiones —filas y columnas— y valores enteros `uint8` entre 0 y 255. Es la misma idea que `color.rgb2gray` de scikit-image, pero con otra escala numérica. Veamos cuánto difieren las dos conversiones.

In [ ]:
globos_gris_skimage = color.rgb2gray(rgb=imagen_globos)

tipo_skimage = globos_gris_skimage.dtype
máximo_skimage = globos_gris_skimage.max()
print(f"scikit-image → tipo: {tipo_skimage} | máximo: {máximo_skimage:.3f}")

# Llevamos scikit-image a la escala 0-255 para poder comparar las dos versiones.
gris_skimage_escalado = globos_gris_skimage * 255
diferencia_media = abs(gris_skimage_escalado - globos_gris_opencv).mean()
print(f"Diferencia media entre las dos conversiones: {diferencia_media:.2f} niveles")

scikit-image devuelve decimales entre 0 y 1; OpenCV, enteros entre 0 y 255. Aun llevando las dos a la misma escala queda una diferencia de varios niveles. No es un error: cada biblioteca **pondera los canales con pesos distintos**.

| Biblioteca | Fórmula de la intensidad |
|---|---|
| OpenCV | 0.299 · R + 0.587 · G + 0.114 · B |
| scikit-image | 0.2125 · R + 0.7154 · G + 0.0721 · B |

En los dos casos el verde pesa más, porque el ojo humano es más sensible a él, pero los coeficientes provienen de estándares de video diferentes.

### ✎ Para pensar

- El cielo de la imagen es azul. ¿Con cuál de las dos fórmulas esperan que el cielo quede más oscuro en grises? Justifiquen con los pesos del canal azul.
- Si un compañero calcula un umbral de brillo con scikit-image y otro con OpenCV, ¿por qué podrían obtener resultados levemente distintos sobre la misma foto?

## 2. Separar y volver a unir canales

**Analogía.** Una canción grabada en un estudio tiene pistas separadas: voz, guitarra y batería. Se puede escuchar cada pista por separado para entender qué aporta, y después volver a mezclarlas. Los canales de una imagen funcionan igual.

**Bajada tecnológica.** Los satélites de observación terrestre capturan decenas de canales, y quienes estudian cultivos o incendios los analizan por separado. En fotografía digital, las correcciones de color se aplican muchas veces canal por canal.

In [ ]:
# 'split' separa la imagen en una lista de canales, en el mismo orden en que están guardados.
canales_de_los_globos = cv2.split(m=imagen_globos)

canal_rojo = canales_de_los_globos[0]
canal_verde = canales_de_los_globos[1]
canal_azul = canales_de_los_globos[2]

cantidad_de_canales = len(canales_de_los_globos)
forma_canal = canal_rojo.shape
print(f"Cantidad de canales: {cantidad_de_canales} | Forma de cada uno: {forma_canal}")

Cada canal es un arreglo de dos dimensiones, del mismo tamaño que la imagen. Como `imagen_globos` ya está en RGB, la posición 0 es el rojo. Si hubiéramos separado la imagen en BGR, la posición 0 sería el azul: `split` no reordena nada, solo separa.

### ✎ Antes de ejecutar

Van a ver cada canal en grises: claro significa mucha presencia de ese color, oscuro significa poca. ¿En qué canal esperan que el cielo se vea más claro? ¿Y los globos amarillos?

In [ ]:
paneles = [
    {"imagen": imagen_globos, "título": "Imagen RGB"},
    {"imagen": canal_rojo, "título": "Canal rojo"},
    {"imagen": canal_verde, "título": "Canal verde"},
    {"imagen": canal_azul, "título": "Canal azul"},
]
mostrar_imágenes(paneles)

El cielo aparece claro en el canal azul y oscuro en el rojo: la luz del cielo es mayormente azul. Los globos amarillos se ven claros en el rojo y en el verde, porque el amarillo de una pantalla se forma sumando luz roja y verde. Los globos rojos solo se destacan en el canal rojo.

Ahora hacemos el camino inverso: unimos los canales con `cv2.merge`. El orden en que los pasamos define qué canal ocupa cada posición de la imagen resultante.

In [ ]:
# 'merge' une una lista de canales en una sola imagen, respetando el orden de la lista.
globos_reconstruidos = cv2.merge(mv=[canal_rojo, canal_verde, canal_azul])
globos_intercambiados = cv2.merge(mv=[canal_azul, canal_verde, canal_rojo])

paneles = [
    {"imagen": globos_reconstruidos, "título": "Unión en orden R, G, B"},
    {"imagen": globos_intercambiados, "título": "Unión en orden B, G, R"},
]
mostrar_imágenes(paneles)

Con el orden original la imagen vuelve a verse igual. Con el rojo y el azul intercambiados, el cielo se vuelve anaranjado: es el mismo efecto que vieron en el cuaderno anterior al mostrar una imagen BGR sin convertir. Ahora saben de dónde viene: es un intercambio de canales.

### ✎ Modificación guiada

Cambien el orden de la lista en `cv2.merge` por `[canal_verde, canal_rojo, canal_azul]` y, antes de ejecutar, anoten de qué color esperan ver el cielo y los globos amarillos.

In [ ]:
orden_de_prueba = [canal_rojo, canal_verde, canal_azul]
globos_de_prueba = cv2.merge(mv=orden_de_prueba)

paneles = [
    {"imagen": imagen_globos, "título": "Original"},
    {"imagen": globos_de_prueba, "título": "Unión con el orden de prueba"},
]
mostrar_imágenes(paneles)

Con el orden original los dos paneles son iguales. Después de modificar la lista, comparen el resultado con su predicción. Si no coincide, vuelvan a mirar los canales separados: cada posición de la lista toma el lugar de un color en la pantalla.

## 3. El espacio HSV

**Analogía.** RGB describe un color como una receta de tres luces: cuánto rojo, cuánto verde, cuánto azul. HSV lo describe como lo haría una persona que pinta: qué pigmento es (matiz), qué tan puro está (saturación) y cuánta luz recibe (valor).

**Bajada tecnológica.** Los selectores de color de los editores de imágenes usan HSV porque es más intuitivo que RGB. En visión por computadora se usa para detectar objetos por su color —semáforos, frutas maduras, pelotas— porque el matiz cambia poco cuando cambia la iluminación.

In [ ]:
# 'COLOR_RGB2HSV' reexpresa cada píxel como matiz, saturación y valor.
globos_hsv = cv2.cvtColor(src=imagen_globos, code=cv2.COLOR_RGB2HSV)
canales_hsv = cv2.split(m=globos_hsv)

canal_matiz = canales_hsv[0]
canal_saturación = canales_hsv[1]
canal_valor = canales_hsv[2]

máximo_matiz = canal_matiz.max()
máximo_saturación = canal_saturación.max()
print(f"Máximo del matiz: {máximo_matiz} | Máximo de la saturación: {máximo_saturación}")

El matiz llega hasta **179** y no hasta 255. El matiz es una posición en un círculo de 360 grados, pero un `uint8` solo guarda hasta 255. OpenCV resuelve el problema dividiendo los grados por dos: el valor 179 equivale a 358 grados.

Para darle sentido a esos números, leemos tres píxeles: uno del cielo, uno de un globo amarillo y uno de un globo rojo. Guardamos cada punto como un diccionario con su nombre y su posición.

In [ ]:
puntos_de_muestra = [
    {"nombre": "cielo", "fila": 50, "columna": 50},
    {"nombre": "globo amarillo", "fila": 330, "columna": 530},
    {"nombre": "globo rojo", "fila": 480, "columna": 650},
]

for punto in puntos_de_muestra:
    nombre = punto["nombre"]
    píxel_hsv = globos_hsv[punto["fila"], punto["columna"]]
    matiz = int(píxel_hsv[0])
    saturación = píxel_hsv[1]
    valor = píxel_hsv[2]
    # Multiplicamos por 2 para volver a la escala de grados del círculo cromático.
    grados = matiz * 2
    print(f"{nombre:>15} → H: {matiz:>3} ({grados:>3} grados) | S: {saturación:>3} | V: {valor:>3}")

El rojo aparece cerca de 0 grados, el amarillo cerca de 45 y el cielo cerca de 200, en la zona de los azules. Los dos globos tienen saturación muy alta: son colores intensos. El cielo también tiene una saturación alta, aunque menor que la de los globos.

### Traducción: mismo concepto, otra escala

Con scikit-image usaron `color.rgb2hsv`, que devuelve los tres canales entre 0 y 1. Esta es la diferencia que más errores produce al pasar de una biblioteca a la otra:

| Canal | scikit-image (`rgb2hsv`) | OpenCV (`COLOR_RGB2HSV`) | Para pasar de scikit-image a OpenCV |
|---|---|---|---|
| H (matiz) | 0.0 a 1.0 | 0 a 179 | multiplicar por 180 |
| S (saturación) | 0.0 a 1.0 | 0 a 255 | multiplicar por 255 |
| V (valor) | 0.0 a 1.0 | 0 a 255 | multiplicar por 255 |

Para comprobarlo, tomamos una condición típica de scikit-image, "saturación mayor a 0.6", y calculamos qué proporción de la imagen la cumple en cada biblioteca.

In [ ]:
umbral_skimage = 0.6
umbral_opencv = umbral_skimage * 255

globos_hsv_skimage = color.rgb2hsv(rgb=imagen_globos)
saturación_skimage = globos_hsv_skimage[:, :, 1]
supera_en_skimage = saturación_skimage > umbral_skimage
supera_en_opencv = canal_saturación > umbral_opencv

proporción_skimage = supera_en_skimage.mean() * 100
proporción_opencv = supera_en_opencv.mean() * 100
print(f"Umbral en OpenCV: {umbral_opencv:.0f}")
print(f"Píxeles saturados → scikit-image: {proporción_skimage:.1f}% | OpenCV: {proporción_opencv:.1f}%")

Con el umbral traducido a la escala 0-255, las dos bibliotecas marcan prácticamente la misma proporción de píxeles. Si hubiéramos copiado el `0.6` sin traducirlo, en OpenCV casi toda la imagen lo superaría, porque allí la saturación llega hasta 255: la condición no tendría ningún sentido.

### ✎ Para pensar

- ¿Por qué el matiz se multiplica por 180 y la saturación por 255, si las dos empiezan en la misma escala de 0 a 1?
- Un píxel gris puro tiene saturación cero. ¿Qué valor de matiz le asignarían ustedes? ¿Tiene sentido hablar de su "tipo de color"?

### ✎ Completado

En scikit-image, el amarillo de los globos tiene un matiz cercano a **0.12**. Traduzcan ese valor a la escala de OpenCV y comprueben si coincide con el matiz que leímos en el píxel del globo amarillo.

In [ ]:
# ─── Completen los pasos faltantes ───────────────────────────────────────────
#
# TODO 1: guarden en 'matiz_amarillo_skimage' el valor 0.12.
# TODO 2: calculen 'matiz_amarillo_opencv' con la regla de la tabla de traducción.
# TODO 3: muestren con print el valor traducido y el matiz leído en el píxel (330, 530).
#
# matiz_amarillo_skimage = ...
# matiz_amarillo_opencv = ...
# matiz_leido = globos_hsv[330, 530][0]
# print(...)
#
# ─────────────────────────────────────────────────────────────────────────────

## 4. Tres representaciones de la misma imagen

Cerramos con el producto del cuaderno: una figura que pone lado a lado la imagen en RGB, en escala de grises y los tres canales de HSV.

In [ ]:
paneles = [
    {"imagen": imagen_globos, "título": "RGB"},
    {"imagen": globos_gris_opencv, "título": "Escala de grises"},
    {"imagen": canal_matiz, "título": "H: matiz (0 a 179)"},
    {"imagen": canal_saturación, "título": "S: saturación"},
    {"imagen": canal_valor, "título": "V: valor"},
]
mostrar_imágenes(paneles)

Cuatro lecturas de la figura:

- **Saturación.** La mayoría de los globos se ve casi blanca, salvo en los reflejos de luz, donde el color se lava y la saturación cae. El cielo queda en un gris claro: también es un color bastante intenso.
- **Valor frente a escala de grises.** Tienen la misma forma, pero V es mucho más claro. V toma el **mayor** de los tres canales, mientras que la escala de grises los **promedia con pesos**. En el píxel del cielo que leímos, V vale 214 y la escala de grises, 140.
- **Matiz.** El panel se ve más oscuro que los demás porque el matiz nunca supera 179, y la figura usa la escala completa de 0 a 255.
- **Globos rojos en el matiz.** Aparecen manchas claras sobre fondo negro. El rojo está justo en el borde del círculo cromático: la mayoría de sus píxeles cae cerca de 0, pero una parte cae cerca de 179. En esta imagen, de los píxeles rojos saturados, unos 7.000 de 81.000 tienen un matiz mayor a 170.

### ✎ Para pensar

- Si quisieran separar los globos del cielo, ¿qué canal de los cinco paneles elegirían como punto de partida? ¿Por qué?
- ¿Qué información se pierde en la escala de grises que sí conserva el canal de matiz?

## Actividad de producción

In [ ]:
# ─── Espacio de práctica ─────────────────────────────────────────────────────
#
# Consigna: analicen los canales de messi.jpg.
#   1. Carguen la imagen con leer_imagen_rgb.
#   2. Conviértanla a HSV y separen sus tres canales.
#   3. Armen una figura con la imagen y los canales H, S y V.
#   4. Escriban en una celda de texto qué canal separa mejor el brazalete
#      verde flúor del resto de la escena, y por qué creen que ocurre.
#
# ─────────────────────────────────────────────────────────────────────────────

## Cierre

Una misma imagen admite varias descripciones numéricas. RGB sirve para mostrarla, la escala de grises para trabajar con la luz, y HSV para razonar sobre el color como lo haría una persona.

| Representación | Canales | Escala en OpenCV | Útil para |
|---|---|---|---|
| RGB | 3 | 0 a 255 | Mostrar la imagen |
| Escala de grises | 1 | 0 a 255 | Texto, bordes, detección |
| HSV | 3 | H: 0 a 179, S y V: 0 a 255 | Razonar y seleccionar por color |

**Limitaciones de lo que vimos.** El matiz pierde sentido en los píxeles casi grises, donde la saturación es muy baja: un pequeño ruido cambia mucho su valor. Además, las conversiones a grises de OpenCV y de scikit-image no son intercambiables, porque usan pesos distintos.

En el próximo cuaderno, `003_archivos`, cambiamos de pregunta: ya no qué representan los números de una imagen, sino cómo se guardan en disco y qué se pierde en el camino.